#### Opinion Review 데이터 세트를 이용한 문서 군집화 수행하기

In [17]:
import glob, os
import pandas as pd

path = '/Users/songbeom/PythonWorkSpace/machineLearning/perfect_guide/8장/topics'
all_files = glob.glob(os.path.join(path, '*.data'))

filename_list = []
opinion_text = []

for file_ in all_files:
    # 개별 파일을 읽어서 DataFrame으로 생성
    df = pd.read_table(file_, index_col=None, header=0, encoding='latin1')

    # 절대경로로 주어진 file 명을 가공 -> 맨 마지막 확장자를 제거해줍니다.
    filename = file_.split('/')[-1].split('.')[0]

    # 파일명 리스트와 파일 내용 리스트에 파일명과 파일 내용을 추가합니다.
    filename_list.append(filename)
    opinion_text.append(df.to_string())

# DataFrame 형태로 생성해줍니다.
document_df = pd.DataFrame({'filename' : filename_list, 'opinion_text' : opinion_text})
document_df.head(5)

,filename,opinion_text
0,battery-life_ipod_nano_8gb,...
1,gas_mileage_toyota_camry_2007,...
2,room_holiday_inn_london,...
3,location_holiday_inn_london,...
4,staff_bestwestern_hotel_sfo,...


In [18]:
all_files = glob.glob(os.path.join(path, '*.data'))
print(all_files[:5])
print("======")
# 파일명 추출 확장자 제거
file_name = all_files[0].split('/')[-1].split('.')[0]
print(file_name)

['/Users/songbeom/PythonWorkSpace/machineLearning/perfect_guide/8장/topics/battery-life_ipod_nano_8gb.txt.data', '/Users/songbeom/PythonWorkSpace/machineLearning/perfect_guide/8장/topics/gas_mileage_toyota_camry_2007.txt.data', '/Users/songbeom/PythonWorkSpace/machineLearning/perfect_guide/8장/topics/room_holiday_inn_london.txt.data', '/Users/songbeom/PythonWorkSpace/machineLearning/perfect_guide/8장/topics/location_holiday_inn_london.txt.data', '/Users/songbeom/PythonWorkSpace/machineLearning/perfect_guide/8장/topics/staff_bestwestern_hotel_sfo.txt.data']
battery-life_ipod_nano_8gb


**TfidfVecorizer의 tokenizer 인자로 사용될 lemmatization 어근 변환 함수를 설정.**

In [19]:
from nltk.stem import WordNetLemmatizer
import nltk
import string

remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)
lemmar = WordNetLemmatizer()

# 입력으로 들어온 token 단어들에 대해서 lemmatization 어근 변환
def LemTokens(tokens):
    return [lemmar.lemmatize(token) for token in tokens]

# 입력으로 문장을 받아서 stop words 제거 -> 소문자 변환 -> 단어 토큰화 -> lemmatization 어근 변환
def LemNormalize(text):
    return LemTokens(nltk.word_tokenize(text.lower().translate(remove_punct_dict)))


#### TF-IDF 기반 Vectorization 적용 및 KMeans 군집화 수행
- Stemmingr과 Lemmatization 같은 어근 변환은 TfidfVectorizer에서 직접 지원하진 않으나 tokenizer 파라미터에 커스팀 어근 변환 함수를 적용하여 어근 변환를 수행할 수 있음
- TfidfVecorizer 생성자의 tokenizer인자로 위에서 생성 LemNormalize 함수 설정

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english',
                             ngram_range=(1, 2), min_df=0.05, max_df=0.85)

feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])

/Users/songbeom/PythonWorkSpace/machineLearning/.venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/songbeom/PythonWorkSpace/machineLearning/.venv/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ha', 'u', 'wa'] not in stop_words.
  warnings.warn(


In [26]:
feature_vect.shape

(51, 4610)

In [28]:
from sklearn.cluster import KMeans

# 5개 집합으로 군집화 수행
km_cluster = KMeans(n_clusters=5, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_
cluster_center = km_cluster.cluster_centers_

In [31]:
print(cluster_label.shape) # 51개의 문서
cluster_center.shape

(51,)


(5, 4610)

In [32]:
document_df['cluster_label'] = cluster_label
document_df.head()

,filename,opinion_text,cluster_label
0,battery-life_ipod_nano_8gb,...,1
1,gas_mileage_toyota_camry_2007,...,4
2,room_holiday_inn_london,...,3
3,location_holiday_inn_london,...,0
4,staff_bestwestern_hotel_sfo,...,3


In [33]:
document_df[document_df['cluster_label'] == 0].sort_values(by='filename')

,filename,opinion_text,cluster_label
17,food_holiday_inn_london,...,0
32,food_swissotel_chicago,...,0
3,location_holiday_inn_london,...,0
41,price_amazon_kindle,...,0
28,price_holiday_inn_london,...,0
16,service_bestwestern_hotel_sfo,...,0
27,service_holiday_inn_london,...,0
13,service_swissotel_hotel_chicago,...,0


In [35]:
document_df[document_df['cluster_label'] == 1].sort_values(by='filename')

,filename,opinion_text,cluster_label
33,accuracy_garmin_nuvi_255W_gps,...,1
9,battery-life_amazon_kindle,...,1
0,battery-life_ipod_nano_8gb,...,1
11,battery-life_netbook_1005ha,...,1
26,buttons_amazon_kindle,...,1
34,directions_garmin_nuvi_255W_gps,...,1
48,display_garmin_nuvi_255W_gps,...,1
36,eyesight-issues_amazon_kindle,...,1
21,features_windows7,...,1
44,fonts_amazon_kindle,...,1


In [37]:
document_df[document_df['cluster_label'] == 2].sort_values(by='filename')

,filename,opinion_text,cluster_label
45,interior_honda_accord_2008,...,2
22,interior_toyota_camry_2007,...,2
42,quality_toyota_camry_2007,...,2


In [38]:
document_df[document_df['cluster_label'] == 3].sort_values(by='filename')


,filename,opinion_text,cluster_label
31,bathroom_bestwestern_hotel_sfo,...,3
49,free_bestwestern_hotel_sfo,...,3
39,location_bestwestern_hotel_sfo,...,3
50,parking_bestwestern_hotel_sfo,...,3
2,room_holiday_inn_london,...,3
46,rooms_bestwestern_hotel_sfo,...,3
30,rooms_swissotel_chicago,...,3
4,staff_bestwestern_hotel_sfo,...,3
20,staff_swissotel_chicago,...,3


In [41]:
document_df[document_df['cluster_label'] == 4].sort_values(by='filename')

,filename,opinion_text,cluster_label
18,comfort_honda_accord_2008,...,4
43,comfort_toyota_camry_2007,...,4
1,gas_mileage_toyota_camry_2007,...,4
35,mileage_honda_accord_2008,...,4
47,performance_honda_accord_2008,...,4
29,seats_honda_accord_2008,...,4
23,transmission_toyota_camry_2007,...,4


In [42]:
from sklearn.cluster import KMeans

# 3개의 집합으로 군집화
km_cluster = KMeans(n_clusters=3, max_iter=10000, random_state=0)
km_cluster.fit(feature_vect)
cluster_label = km_cluster.labels_

document_df['cluster_label'] = cluster_label
document_df.sort_values(by='cluster_label')

,filename,opinion_text,cluster_label
50,parking_bestwestern_hotel_sfo,...,0
27,service_holiday_inn_london,...,0
28,price_holiday_inn_london,...,0
30,rooms_swissotel_chicago,...,0
20,staff_swissotel_chicago,...,0
31,bathroom_bestwestern_hotel_sfo,...,0
32,food_swissotel_chicago,...,0
17,food_holiday_inn_london,...,0
16,service_bestwestern_hotel_sfo,...,0
13,service_swissotel_hotel_chicago,...,0


**군집(Cluster)별 핵심 단어 추출하기**

In [45]:
cluster_centers = km_cluster.cluster_centers_
print(cluster_centers.shape)
print(cluster_centers)

(3, 4610)
[[0.         0.00099548 0.00174656 ... 0.         0.00183397 0.00144581]
 [0.0100545  0.         0.         ... 0.00706288 0.         0.        ]
 [0.         0.00092552 0.         ... 0.         0.         0.        ]]


In [47]:
# 군집별 top n 핵심단어, 그 단어의 중심 위치 상대값, 대상 파일명들을 반환
def get_cluster_details(cluster_model, cluster_data, feature_names, clusters_num, top_n_features=10):
    cluster_details = {}

    # cluster_centers array의 값이 큰 순으로 정렬된 index 값을 반환
    # 군집 중심점(centroid)별 할당된 word 피처들의 거리값이 큰 순으로 값을 구하기 위함.
    centroid_feature_ordered_ind = cluster_model.cluster_centers_.argsort()[::-1]

    for cluster_num in range(clusters_num):
        cluster_details[cluster_num] = {}
        cluster_details[cluster_num]['cluster'] = cluster_num

        top_feature_indexes = centroid_feature_ordered_ind[cluster_num, :top_n_features]
        top_features = [ feature_names[ind] for ind in top_feature_indexes ]

        top_feature_values = cluster_model.cluster_centers_[cluster_num, top_feature_indexes].tolist()

        cluster_details[cluster_num]['top_features'] = top_features
        cluster_details[cluster_num]['top_features_values'] = top_feature_values
        filenames = cluster_data[cluster_data['cluster_label'] == cluster_num]['filename']
        filenames = filenames.values.tolist()
        cluster_details[cluster_num]['filenames'] = filenames
    return cluster_details


In [48]:
def print_cluster_details(cluster_details):
    for cluster_num, cluster_detail in cluster_details.items():
        print('###### Cluster {0}'.format(cluster_num))
        print('Top features:', cluster_detail['top_features'])
        print('Reviews 파일명:', cluster_detail['filenames'][:7])
        print("="*20)

In [49]:
feature_names = tfidf_vect.get_feature_names_out()

cluster_details = get_cluster_details(cluster_model=km_cluster, cluster_data=document_df,
                                      feature_names=feature_names, clusters_num=3, top_n_features=10)

print_cluster_details(cluster_details)

###### Cluster 0
Top features: ['0 5', 'near kensington', 'near tube', 'nearby', 'nearest', 'neat clean', 'necessary', 'necklace', 'necklace earring', 'need direction']
Reviews 파일명: ['room_holiday_inn_london', 'location_holiday_inn_london', 'staff_bestwestern_hotel_sfo', 'service_swissotel_hotel_chicago', 'service_bestwestern_hotel_sfo', 'food_holiday_inn_london', 'staff_swissotel_chicago']
###### Cluster 1
Top features: ['£6', 'cylinder', 'cyl engine', 'cyl', 'right street', 'housekeeper', 'housekeeping', 'housekeeping service', 'cushion', 'curtain']
Reviews 파일명: ['battery-life_ipod_nano_8gb', 'voice_garmin_nuvi_255W_gps', 'speed_garmin_nuvi_255W_gps', 'size_asus_netbook_1005ha', 'screen_garmin_nuvi_255W_gps', 'battery-life_amazon_kindle', 'satellite_garmin_nuvi_255W_gps']
###### Cluster 2
Top features: ['0 5', 'ipod gain', 'iphone', 'intrusive', 'interstate', 'interior trim', 'interior roomy', 'interior quality', 'interior nice', 'interior new']
Reviews 파일명: ['gas_mileage_toyota_camr